In [ ]:
📓 day-2a-Agent_Tools.ipynb
│
├── 📦 SECTION 1: Setup (API Key, Imports)
│   └── Cell 1: Load API Key from .env
│   └── Cell 2: Imports - ALL ADK Components & verify
│   └── Cell 3: BuiltInCodeExecutor Helper function
│   └── Cell 4: Retrying request
│
├── 🔧 SECTION 2: Custom Tools (Currency Agent -> Fee from banks + Exchange Rate) 
│   └── Cell 5:   Fee Lookup Tool
│   └── Cell 5.1: Get exchange rate
│   └── Cell 3: 
│   └── Cell 4: 
│
│
│
├── 🤖 SECTION 3: Agent WITHOUT Code Execution (Original)
│   └── Cell 6: currency_agent (original)—Create Currency Agent, agent is configured but not yet running.
│   └── Cell 7: Quick Test (No Input Required)  - Test with convert_currency()
│   └── Cell 8: WIDGET VERSION (RELIABLE - NO ASYNC ISSUES!)
│
├── 💻 SECTION 9: Agent WITH Code Execution (NEW!)
│   ├── Cell 9:  # Import BuiltInCodeExecutor
│   ├── Cell 10: # Create code_executor
│   ├── Cell 11: currency_agent_with_code (NEW agent)—# Agent WITH Code Generation (4 DECIMAL PRECISION)-generate code itself by LLM
│   ├── Cell 12: Test with convert_currency_with_code() - # Test Agent WITH Code Generation (Manual Execution)
│   └── Cell 13: Compare results side-by-side
│
├── 🎨 SECTION 5: Interactive UI (Widgets)
│   └── Cell 14: Widget for Original Agent
│   └── Cell 15: Widget for Code Execution Agent
│
└── 📊 SECTION 6: Comparison & Analysis
    └── Cell 16: Show differences

In [1]:
    
    # access the GOOGLE_API_KEY just saved and set it as an environment variable for the notebook to use:

    # instead of using kaggle secrete, I switch it to VS Code and do it here .env

    # Cell 1: Load API Key from .env (VS Code / Local Development)

import os
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if GOOGLE_API_KEY:
    print("✅ Gemini API key loaded successfully from .env")
    print(f"   Key starts with: {GOOGLE_API_KEY[:8]}...")
else:
    print("❌ ERROR: GOOGLE_API_KEY not found in .env file")


✅ Gemini API key loaded successfully from .env
   Key starts with: AQ.Ab8RN...


In [2]:
    #  import the specific components you'll need from the Agent Development Kit and the Generative AI library. This keeps your code organized and ensures we have access to the necessary building blocks.

    # Cell 2: Imports - ALL ADK Components

from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor

print("✅ ADK components imported successfully.")

# Cell 2.5: Verify Imports
print("✅ Checking imports...")
print(f"   LlmAgent: {LlmAgent}")
print(f"   Gemini: {Gemini}")
print(f"   InMemoryRunner: {InMemoryRunner}")
print(f"   InMemorySessionService: {InMemorySessionService}")
print("\n✅ All imports successful!")

✅ ADK components imported successfully.
✅ Checking imports...
   LlmAgent: <class 'google.adk.agents.llm_agent.LlmAgent'>
   Gemini: <class 'google.adk.models.google_llm.Gemini'>
   InMemoryRunner: <class 'google.adk.runners.InMemoryRunner'>
   InMemorySessionService: <class 'google.adk.sessions.in_memory_session_service.InMemorySessionService'>

✅ All imports successful!


In [3]:
    # Helper function that prints the generated Python code and results from the code execution tool:
    # show_python_code_and_result() is a helper for ADK's BuiltInCodeExecutor, if I am using it.  In this case, I manually wrote the extract_and_run_code() which is similar in functions
    # Cell 3: BuiltInCodeExecutor Helper function
def show_python_code_and_result(response):
    for i in range(len(response)):
        # Check if the response contains a valid function call result from the code executor
        if (
            (response[i].content.parts)
            and (response[i].content.parts[0])
            and (response[i].content.parts[0].function_response)
            and (response[i].content.parts[0].function_response.response)
        ):
            response_code = response[i].content.parts[0].function_response.response
            if "result" in response_code and response_code["result"] != "```":
                if "tool_code" in response_code["result"]:
                    print(
                        "Generated Python Code >> ",
                        response_code["result"].replace("tool_code", ""),
                    )
                else:
                    print("Generated Python Response >> ", response_code["result"])


print("✅ Helper functions defined.")


✅ Helper functions defined.


In [4]:
    # working with LLMs, encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

    # Cell 4: Retrying request
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

In [5]:
"""
        Building Custom Function Tools ¶
        Example: Currency Converter Agent
        This agent can convert currency from one denomination to another and calculates the fees to do the conversion. The agent has two custom tools and follows the workflow:

        Fee Lookup Tool - Finds transaction fees for the conversion (mock)
        Exchange Rate Tool - Gets currency conversion rates (mock)
        Calculation Step - Calculates the total conversion cost including the fees


        🏆 ADK Best Practices in Action¶
        Notice how our tools follow ADK best practices:

        1. Dictionary Returns: Tools return {"status": "success", "data": ...} or {"status": "error", "error_message": ...}
        2. Clear Docstrings: LLMs use docstrings to understand when and how to use tools
        3. Type Hints: Enable ADK to generate proper schemas (str, dict, etc.)
        4. Error Handling: Structured error responses help LLMs handle failures gracefully

        
                                👤 User
                                    │
                                    ▼
                            🤖 Currency Agent
                                /        \
                                /          \
                            ▼            ▼
                🔧 get_fee_for_        💱 get_exchange_rate
                    payment_method

"""
    # Pay attention to the docstring, type hints, and return value.

    # Cell 5: Fee Lookup Tool
    
def get_fee_for_payment_method(method: str) -> dict:
    """Looks up the transaction fee percentage for a given payment method.

    Args:
        method: The name of the payment method.
                e.g., "platinum credit card" or "bank transfer".

    Returns:
        Dictionary with status and fee information.
        Success: {"status": "success", "fee_percentage": 0.02}
        Error: {"status": "error", "error_message": "Payment method not found"}
    """
    fee_database = {
        "platinum credit card": 0.02, # this can be change to dynamic real world database later
        "gold debit card": 0.035,
        "bank transfer": 0.01,
    }

    fee = fee_database.get(method.lower())
    if fee is not None:
        return {"status": "success", "fee_percentage": fee}
    else:
        return {"status": "error", "error_message": f"Payment method '{method}' not found"}

print("✅ Fee lookup function created")
print(f"💳 Test: {get_fee_for_payment_method('platinum credit card')}")


✅ Fee lookup function created
💳 Test: {'status': 'success', 'fee_percentage': 0.02}


In [6]:
    # to define our second tool get_exchange_rate
    # Cell 5.1 get exchange rate

def get_exchange_rate(base_currency: str, target_currency: str) -> dict:
    """Looks up and returns the exchange rate between two currencies.

    Args:
        base_currency: The ISO 4217 currency code of the currency you
                       are converting from (e.g., "USD").
        target_currency: The ISO 4217 currency code of the currency you
                         are converting to (e.g., "EUR").

    Returns:
        Dictionary with status and rate information.
        Success: {"status": "success", "rate": 0.93}
        Error: {"status": "error", "error_message": "Unsupported currency pair"}
    """

    # Static data simulating a live exchange rate API
    # In production, this would call something like: requests.get("api.exchangerates.com")
    rate_database = {
    "usd": {
        "eur": 0.93,  # will connect to dynamic database later
        "jpy": 157.50,
        "inr": 83.58,
        "gbp": 0.79,  # British Pound
        "cad": 1.36,  # Canadian Dollar
        "aud": 1.52,  # Australian Dollar
        "chf": 0.88,  # Swiss Franc
        "cny": 7.24,  # Chinese Yuan
        "brl": 5.48,  # Brazilian Real
    },
    "eur": {
        "usd": 1.08,
        "jpy": 169.35,
        "inr": 89.87,
        "gbp": 0.85,
        "cad": 1.46,
        "aud": 1.63,
    },
    "jpy": {
        "usd": 0.00635,
        "eur": 0.00590,
        "inr": 0.53,
    }
}

    # Input validation and processing
    base = base_currency.lower()
    target = target_currency.lower()

    rate = rate_database.get(base, {}).get(target)
    if rate is not None:
        return {"status": "success", "rate": rate}
    else:
        return {"status": "error", "error_message": f"Unsupported currency pair: {base_currency}/{target_currency}"}

print("✅ Exchange rate function created")
print(f"💱 Test: {get_exchange_rate('USD', 'EUR')}")


✅ Exchange rate function created
💱 Test: {'status': 'success', 'rate': 0.93}


In [7]:
"""
         create our currency agent. Pay attention to how the agent's instructions reference the tools:

        Key Points:

        The tools=[] list tells the agent which functions it can use
        Instructions reference tools by their exact function names (e.g., get_fee_for_payment_method())
        The agent uses these names to decide when and how to call each tool
"""

        # Cell 6: Create Currency Agent, agent is configured but not yet running.

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner

currency_agent = LlmAgent(
    name="currency_agent",
    model=Gemini(model="gemini-3.1-flash-lite", retry_options=retry_config),
    instruction="""You are a smart currency conversion assistant.

    For currency conversion requests:
    1. Use `get_fee_for_payment_method()` to find transaction fees
    2. Use `get_exchange_rate()` to get currency conversion rates
    3. Check the "status" field in each tool's response for errors
    4. Calculate the final amount after fees and provide a clear breakdown.
    5. First, state the final converted amount.
       Then, explain how you got that result showing: fee percentage, fee amount, amount after fee, and exchange rate used.

    If any tool returns status "error", explain the issue to the user clearly.
    """,
    tools=[get_fee_for_payment_method, get_exchange_rate],
)

print("✅ Currency agent created with custom function tools")

✅ Currency agent created with custom function tools


In [12]:
    # Cell 7: Quick Test (No Input Required)  - Test with convert_currency()
import asyncio
import nest_asyncio

nest_asyncio.apply()

    # Create runner
currency_runner = InMemoryRunner(agent=currency_agent)

async def convert_currency(amount: float, base_currency: str, target_currency: str, payment_method: str):
    """Convert currency with the agent."""
    query = f"Convert {amount} {base_currency.upper()} to {target_currency.upper()} using {payment_method.lower()}"
    
    print("\n" + "-"*60)
    print(f"🤖 Processing: {query}")
    print("-"*60 + "\n")

        # The LLM (gemini-3.1-flash-lite) is only called when you run the agent:
        # # THIS is when the LLM wakes up!
    response = await currency_runner.run_debug(query)
    
    for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)
    
    print("\n" + "✅ Conversion complete!")

# ===== TEST WITH PRE-FILLED VALUES =====
print("="*60)
print("💱 QUICK TEST: 79.03 USD to EUR with Platinum Credit Card")
print("="*60)

await convert_currency(79.03, "USD", "EUR", "platinum credit card")

💱 QUICK TEST: 79.03 USD to EUR with Platinum Credit Card

------------------------------------------------------------
🤖 Processing: Convert 79.03 USD to EUR using platinum credit card
------------------------------------------------------------

currency_agent > The final converted amount is **71.93 EUR**.

Here is the breakdown of the calculation:

*   **Initial Amount:** 79.03 USD
*   **Fee Percentage:** 2% (platinum credit card)
*   **Fee Amount:** 1.58 USD (79.03 * 0.02)
*   **Amount After Fee:** 77.45 USD (79.03 - 1.58)
*   **Exchange Rate Used:** 0.93 (USD to EUR)
*   **Final Amount:** 72.03 EUR (77.45 * 0.93)

*(Note: There was a minor rounding adjustment in the final step; calculation based on exact values results in 72.0285 EUR).*
The final converted amount is **71.93 EUR**.

Here is the breakdown of the calculation:

*   **Initial Amount:** 79.03 USD
*   **Fee Percentage:** 2% (platinum credit card)
*   **Fee Amount:** 1.58 USD (79.03 * 0.02)
*   **Amount After Fee:** 77.45 

In [13]:

    # Cell 8: WIDGET VERSION (RELIABLE - NO ASYNC ISSUES!)
import ipywidgets as widgets
from IPython.display import display, clear_output
import asyncio
import nest_asyncio
import sys
from io import StringIO

nest_asyncio.apply()

# Create runner
currency_runner = InMemoryRunner(agent=currency_agent)

# ===== THE CONVERSION FUNCTION (SYNCHRONOUS WRAPPER) =====
def convert_currency_sync(amount, base, target, payment):
    """Synchronous wrapper for the async conversion function."""
    
    # Create a new event loop for this operation
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    try:
        query = f"Convert {amount} {base.upper()} to {target.upper()} using {payment.lower()}"
        
        # Capture ALL print output
        captured = StringIO()
        original_stdout = sys.stdout
        sys.stdout = captured
        
        try:
            print("-"*60)
            print(f"🤖 Processing: {query}")
            print("-"*60 + "\n")
            
            # Run the async function synchronously
            response = loop.run_until_complete(currency_runner.run_debug(query))
            
            for event in response:
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            print(part.text)
            
            print("\n" + "✅ Conversion complete!")
        except Exception as e:
            print(f"❌ Error: {e}")
        finally:
            sys.stdout = original_stdout
        
        return captured.getvalue()
        
    finally:
        loop.close()

# ===== CREATE WIDGETS =====
amount_input = widgets.FloatText(
    value=500.0,
    description='Amount:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

base_input = widgets.Dropdown(
    options=['USD', 'EUR', 'JPY', 'GBP', 'INR'],
    value='USD',
    description='From:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='120px')
)

target_input = widgets.Dropdown(
    options=['USD', 'EUR', 'JPY', 'GBP', 'INR'],
    value='EUR',
    description='To:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='120px')
)

payment_input = widgets.Dropdown(
    options=['platinum credit card', 'gold debit card', 'bank transfer'],
    value='platinum credit card',
    description='Payment:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# ===== OUTPUT AREA =====
output = widgets.Output(
    layout=widgets.Layout(
        border='2px solid #4CAF50',
        padding='15px',
        margin='10px 0px',
        max_height='500px',
        overflow='auto',
        background_color='#f9f9f9',
        border_radius='5px',
        width='100%'
    )
)

# ===== CONVERT BUTTON =====
convert_button = widgets.Button(
    description='🔄 Convert!',
    button_style='success',
    layout=widgets.Layout(width='150px', margin='10px 0px')
)

# ===== HANDLE BUTTON CLICK (SYNCHRONOUS) =====
def on_convert(b):
    # Clear previous output
    output.clear_output()
    
    # Get values
    amount = amount_input.value
    base = base_input.value
    target = target_input.value
    payment = payment_input.value
    
    # Show working message
    with output:
        print("⏳ Processing your conversion... Please wait...\n")
    
    # Run the conversion SYNCHRONOUSLY (no async issues!)
    result = convert_currency_sync(amount, base, target, payment)
    
    # Display the result
    with output:
        clear_output()
        print(result)

convert_button.on_click(on_convert)

# ===== DISPLAY UI =====
print("\n" + "="*60)
print("💱 CURRENCY CONVERTER WITH FEE CALCULATOR")
print("="*60)
print("\n📝 Fill in the fields below and click 'Convert!'\n")

# Create a nice layout with full width
ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([
            widgets.HBox([widgets.Label(value='💰', layout=widgets.Layout(width='40px')), amount_input]),
            widgets.HBox([widgets.Label(value='💵 From:', layout=widgets.Layout(width='40px')), base_input]),
            widgets.HBox([widgets.Label(value='💶 To:', layout=widgets.Layout(width='40px')), target_input]),
            widgets.HBox([widgets.Label(value='💳', layout=widgets.Layout(width='40px')), payment_input]),
            convert_button,
        ], layout=widgets.Layout(padding='10px', align_items='flex-start')),
    ]),
    output,
], layout=widgets.Layout(width='100%', align_items='center'))

display(ui)

print("\n✅ UI Ready! Fill in the fields and click 'Convert!'")


💱 CURRENCY CONVERTER WITH FEE CALCULATOR

📝 Fill in the fields below and click 'Convert!'




✅ UI Ready! Fill in the fields and click 'Convert!'


In [ ]:
    # Cell 9: Import BuiltInCodeExecutor

from google.adk.code_executors import BuiltInCodeExecutor

print("✅ Code Executor imported successfully!")

In [ ]:
    # Cell 10: Create code_executor
code_executor = BuiltInCodeExecutor()

print("✅ Code Executor created!")
print(f"   Type: {type(code_executor)}")

In [14]:
# Cell 11: Agent WITH Code Generation (4 DECIMAL PRECISION)-Agent itself create by LLM
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini

currency_agent_with_code = LlmAgent(
    name="currency_agent_with_code",
    model=Gemini(model="gemini-3.1-flash-lite", retry_options=retry_config),
    instruction="""You are a smart currency conversion assistant.

    For currency conversion requests:
    1. Use `get_fee_for_payment_method()` to find transaction fees
    2. Use `get_exchange_rate()` to get currency exchange rates
    3. Check the "status" field in each tool's response for errors
    4. **IMPORTANT: Generate Python code to do the math, but don't execute it.**
    5. Generate Python code that:
       - Takes the fee percentage and exchange rate from the tool responses
       - Calculates: fee_amount = amount * fee_percentage
       - Calculates: amount_after_fee = amount - fee_amount
       - Calculates: final_amount = amount_after_fee * exchange_rate
       - Prints the breakdown with **4 decimal places** (e.g., {fee_amount:.4f})
       - Prints the final amount with **4 decimal places** (e.g., {final_amount:.4f})
    6. Output the Python code in a clear markdown code block.
    7. Then explain the result in plain English.

    If any tool returns status "error", explain the issue to the user clearly.
    """,
    tools=[get_fee_for_payment_method, get_exchange_rate],
)

print("✅ Currency agent created (will generate Python code with 4 decimal precision)")

✅ Currency agent created (will generate Python code with 4 decimal precision)


In [16]:

    # Cell 12: Test Agent WITH Code Generation (Manual Execution)

import asyncio
import nest_asyncio
import re
import subprocess
import tempfile
import os

nest_asyncio.apply()

# Create a separate runner for the new agent
runner_with_code = InMemoryRunner(agent=currency_agent_with_code)

def extract_and_run_code(text: str) -> str:
    """Extract Python code from markdown and run it."""
    # Look for code blocks
    code_pattern = r'```python\n(.*?)\n```'
    matches = re.findall(code_pattern, text, re.DOTALL)
    
    if not matches:
        code_pattern = r'```\n(.*?)\n```'
        matches = re.findall(code_pattern, text, re.DOTALL)
    
    if not matches:
        return "No code block found in response."
    
    # Get the code and strip any extra indentation
    code = matches[0]
    
    # ✅ FIX: Properly indent the user's code
    lines = code.split('\n')
    
    # Find minimum indentation (to remove common leading whitespace)
    min_indent = float('inf')
    for line in lines:
        if line.strip():
            indent = len(line) - len(line.lstrip())
            min_indent = min(min_indent, indent)
    
    # Remove common indentation from all lines
    if min_indent != float('inf'):
        lines = [line[min_indent:] if line.strip() else line for line in lines]
    
    # ✅ CRITICAL FIX: Add 4 spaces to EVERY line so it's inside the try block
    indented_lines = ['    ' + line if line.strip() else line for line in lines]
    indented_code = '\n'.join(indented_lines)
    
    # Build the wrapper WITHOUT indentation issues
    full_code = f"""import sys
from io import StringIO

out = StringIO()
sys.stdout = out

try:
{indented_code}
except Exception as e:
    print(f"Error: {{e}}")

sys.stdout = sys.__stdout__
print(out.getvalue())
"""
    
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(full_code)
            temp_file = f.name
        
        result = subprocess.run(
            ['python', temp_file],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        os.unlink(temp_file)
        
        if result.returncode == 0:
            return result.stdout
        else:
            return f"Error running code:\n{result.stderr}"
            
    except subprocess.TimeoutExpired:
        return "Code execution timed out."
    except Exception as e:
        return f"Error: {e}"

async def convert_currency_with_code(amount: float, base_currency: str, target_currency: str, payment_method: str):
    """Convert currency using the agent WITH code generation."""
    query = f"Convert {amount} {base_currency.upper()} to {target_currency.upper()} using {payment_method.lower()}"
    
    print("\n" + "="*60)
    print("🧪 TESTING AGENT WITH CODE GENERATION")
    print("="*60)
    print("-"*60)
    print(f"🤖 Processing: {query}")
    print("-"*60 + "\n")
    
    response = await runner_with_code.run_debug(query)
    
    full_text = ""
    for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(part.text)
                    full_text += part.text
    
    # Try to extract and run code from the response
    print("\n" + "-"*40)
    print("📝 Attempting to extract and run code...")
    print("-"*40)
    
    code_result = extract_and_run_code(full_text)
    if code_result and "No code block found" not in code_result:
        print("\n✅ Code Execution Result:")
        print(code_result)
    else:
        print("\nℹ️ No executable code block found in response.")
    
    print("\n" + "✅ Conversion complete with code generation!")

# Run the test
await convert_currency_with_code(79.03, "USD", "EUR", "platinum credit card")



🧪 TESTING AGENT WITH CODE GENERATION
------------------------------------------------------------
🤖 Processing: Convert 79.03 USD to EUR using platinum credit card
------------------------------------------------------------

currency_agent_with_code > To convert 79.03 USD to EUR using a platinum credit card, we first account for the 2.0% transaction fee and then apply the exchange rate of 0.93.

Here is the Python code to perform this calculation:

```python
amount = 79.03
fee_percentage = 0.02
exchange_rate = 0.93

# Calculations
fee_amount = amount * fee_percentage
amount_after_fee = amount - fee_amount
final_amount = amount_after_fee * exchange_rate

# Display results
print(f"Fee amount: {fee_amount:.4f}")
print(f"Amount after fee: {amount_after_fee:.4f}")
print(f"Final converted amount: {final_amount:.4f}")
```

### Explanation:
1.  **Fee Calculation:** A 2% fee on 79.03 USD results in a deduction of **1.5806 USD**.
2.  **Net Amount:** After subtracting the fee, you have **77.449

In [ ]:
# Cell 13: Compare Original vs Code Execution
print("\n" + "="*60)
print("📊 SIDE-BY-SIDE COMPARISON")
print("="*60)

print("""
┌─────────────────────────────────────────────────────────────┐
│                    WITHOUT CODE EXECUTION                   │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  LLM guesses the math                              │   │
│  │  ⚠️ Potential rounding errors                      │   │
│  │  ⚠️ Inconsistent results                           │   │
│  │  ⚠️ Black box - can't see the math                 │   │
│  └─────────────────────────────────────────────────────┘   │
├─────────────────────────────────────────────────────────────┤
│                    WITH CODE EXECUTION                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Python computes the math                          │   │
│  │  ✅ Always exact                                   │   │
│  │  ✅ Always consistent                              │   │
│  │  ✅ Code is visible and auditable                  │   │
│  └─────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
""")